# Ácidos: ionização e pH
### Química — funções inorgânicas

**Objetivo:** Reconhecer ácidos, nomeá-los e calcular o pH de uma solução ideal de HCl.

### Como usar este notebook
Execute as células em ordem. As de código conferem as contas e geram as visualizações. No 3D, **arraste com o mouse para girar em 360°** e use a roda para zoom.
A primeira célula instala somente dependências ausentes. O 3D precisa de acesso a `cdn.jsdelivr.net` no navegador e de um notebook confiável (trusted); se a rede bloquear o visualizador, o desenho 2D continua disponível.

Material para estudo teórico: não prove, toque nem misture produtos químicos para identificar uma função química.

In [ ]:
# Preparação idempotente: usa o Python do próprio kernel.
import importlib.util
import subprocess
import sys

for pacote, modulo in [("rdkit", "rdkit"), ("py3Dmol", "py3Dmol"),
                       ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pacote], check=True)

import math
import matplotlib.pyplot as plt
import py3Dmol
from rdkit import Chem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import SVG, display

## 1. Conceito principal

Na definição de **Arrhenius**, ácidos aumentam a concentração de $\mathrm{H_3O^+}$ em água. Na de **Brønsted–Lowry**, são doadores de prótons. O próton não fica livre em água: $\mathrm{H^+}$ é uma abreviação comum de sua forma hidratada.

$$\mathrm{HCl + H_2O \rightarrow H_3O^+ + Cl^-}$$
$$\boxed{\mathrm{pH} \approx -\log_{10}([H_3O^+]/(1\ \mathrm{mol/L}))}$$

A expressão com concentração aproxima a definição rigorosa por atividade, adequada a soluções diluídas. Adotamos 25 °C.

| Critério | Exemplos |
|---|---|
| Hidrácidos (sem oxigênio) | HCl: ácido clorídrico; H₂S: ácido sulfídrico |
| Oxiácidos | HNO₃: ácido nítrico; H₂SO₄: ácido sulfúrico |
| Fortes / fracos | HCl ioniza praticamente por completo; HF ioniza parcialmente |

**Força não é concentração.** A quantidade de H na fórmula não é necessariamente o número de prótons ionizáveis: H₃PO₃ é diprótico. Não use a aproximação de ácido forte para qualquer ácido.

## 2. Exemplo resolvido passo a passo

Uma solução ideal de HCl tem concentração $0{,}010\ \mathrm{mol/L}$. Por ser forte e monoprótico, a contribuição da água é desprezível nesta concentração.

$$[H_3O^+]\approx c=1{,}0\times10^{-2}\ \mathrm{mol/L}$$
$$\mathrm{pH}=-\log_{10}(10^{-2})=\boxed{2{,}00}$$

## 3. Conferindo o cálculo com Python

In [ ]:
c = 0.010  # mol/L
ph = -math.log10(c)
print(f"[H3O+] ≈ {c:.3f} mol/L; pH = {ph:.2f}")
assert math.isclose(ph, 2.0)

## 4. Visualização

### 4.1 Modelo 3D interativo

**Cloreto de hidrogênio (HCl), antes da ionização.** Cl = verde, H = branco. A distância ilustrativa é 1,27 Å. Em água, não interprete este desenho como se todo HCl permanecesse molecular.

In [ ]:
# Coordenadas didáticas em Å; não são uma simulação de solução.
SMILES = 'Cl'
COORDENADAS = [(0.0, 0.0, 0.0), (1.27, 0.0, 0.0)]
mol = Chem.MolFromSmiles(SMILES)
if mol is None:
    raise ValueError(f"SMILES inválido: {SMILES!r}")
mol = Chem.AddHs(mol)
if mol.GetNumAtoms() != len(COORDENADAS):
    raise ValueError("O número de coordenadas não corresponde ao de átomos.")
conf = Chem.Conformer(mol.GetNumAtoms())
for indice, posicao in enumerate(COORDENADAS):
    conf.SetAtomPosition(indice, posicao)
mol.AddConformer(conf)
visor = py3Dmol.view(width=640, height=400)
visor.addModel(Chem.MolToMolBlock(mol), "mol")
visor.setStyle({"stick": {"radius": 0.13, "colorscheme": "Jmol"},
                 "sphere": {"scale": 0.28, "colorscheme": "Jmol"}})
visor.setBackgroundColor("#ffffff")
visor.zoomTo()
visor.show()

### 4.2 Representação 2D

Cada traço representa uma ligação; dois traços representam uma ligação dupla.

In [ ]:
# Estrutura 2D: ligações covalentes com traços; íons separados por ponto
# no SMILES permanecem separados, sem inventar uma ligação covalente.
plano = Chem.Mol(mol)
rdDepictor.Compute2DCoords(plano)
desenho = rdMolDraw2D.MolDraw2DSVG(520, 300)
desenho.drawOptions().bondLineWidth = 2.2
desenho.DrawMolecule(plano)
desenho.FinishDrawing()
display(SVG(desenho.GetDrawingText()))

### 4.3 Gráfico de dados

In [ ]:
cs = [10**(-k / 20) for k in range(20, 81)]
plt.plot(cs, [-math.log10(c) for c in cs])
plt.xscale("log")
plt.xlabel("Concentração de HCl (mol/L)")
plt.ylabel("pH (aproximação ideal)")
plt.title("Ácido forte monoprótico: concentração e pH")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Como interpretar o resultado

O traço H–Cl representa uma ligação covalente na espécie molecular, não uma ligação entre íons em solução. Diluir dez vezes eleva o pH em uma unidade no intervalo do gráfico. Abaixo de concentrações muito pequenas, é preciso considerar a autoionização da água.

## 6. Exercício de fixação

Uma solução ideal de HCl a 0,001 mol/L tem qual pH? Antes de executar, preveja a variação em relação ao exemplo.

Tente resolver antes de executar a resposta abaixo.

In [ ]:
c_ex = 0.001  # mol/L
ph_ex = -math.log10(c_ex)
print(f"[H3O+] ≈ {c_ex:.3f} mol/L; pH = {ph_ex:.2f}")
assert math.isclose(ph_ex, 3.0)

## 7. Resumo

- Ácidos de Arrhenius aumentam H₃O⁺ em água.
- Hidrácidos e oxiácidos se distinguem pela presença de oxigênio.
- Força e concentração são conceitos diferentes.
- Para HCl diluído, pH ≈ −log₁₀(c), nas condições adotadas.

## Referências

- IUPAC. *Compendium of Chemical Terminology (Gold Book)*: https://goldbook.iupac.org/
- ATKINS, P.; JONES, L. *Princípios de Química*. Bookman.